# E7 — Security Checks / Defenses: reproducing the paper's Table 1, in text
Rows = defenses. Columns = attack configs, mapped from the paper's image triggers onto our text triggers:

| Paper's column | Our column |
|---|---|
| BadNet + Random | word_random (E2-word) |
| BadNet + CBS | word_cbs (E3-word) |
| Blend + Random | sent_random (E2-sent) |
| Blend + CBS | sent_cbs (E3-sent) |

Values = **ASR of a model retrained from scratch on the defense-filtered training set** -- this is the same protocol the paper itself uses, not just a raw detection rate.

**Important scope note (read before trusting the numbers):** this table only makes sense for the **teachers** (E2/E3). The **students** (E5/E6) are distilled on clean, untriggered data -- there are no poisoned training examples in their training set for a data-filtering defense to find. Section 6 below covers what a meaningful security check looks like for the students instead (inference-time STRIP).

**Prerequisites:** run `e1.ipynb`, `random_poisoning.ipynb`, `e3_cbs.ipynb` first (this notebook reloads their saved checkpoints and regenerates their poisoned training sets deterministically from the same seed).

In [1]:
!pip install transformers datasets scikit-learn scipy --quiet



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random, json as pyjson, os
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                           TrainingArguments, Trainer, GPT2LMHeadModel, GPT2TokenizerFast)
from sklearn.metrics import accuracy_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "bert-base-uncased"
MAX_LEN = 64
TARGET_LABEL = 1
POISON_RATE_WORD_RANDOM = 0.0006
POISON_RATE_WORD_CBS    = 0.005
POISON_RATE_SENT_RANDOM = 0.0004
POISON_RATE_SENT_CBS    = 0.002
WORD_TRIGGER = "cf"
SENT_TRIGGER = "The absent gerbil filed a complaint downtown."
# how many examples to run the (expensive) detection defenses over -- all poisoned examples are
# always included, plus a random clean sample. Raise this for your final reported numbers.
DEFENSE_SAMPLE_SIZE = 3000
# epochs used when RETRAINING on the filtered set for the ASR table -- raise to 3 for final numbers
RETRAIN_EPOCHS = 3
print(DEVICE)

cuda


In [3]:
ds = load_dataset("stanfordnlp/sst2")
clean_train_df = pd.DataFrame({"sentence": ds["train"]["sentence"], "label": ds["train"]["label"]})
clean_valid_df = pd.DataFrame({"sentence": ds["validation"]["sentence"], "label": ds["validation"]["label"]})
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    d = Dataset.from_pandas(df[["sentence", "label"]].reset_index(drop=True))
    d = d.map(lambda b: tokenizer(b["sentence"], truncation=True, padding="max_length", max_length=MAX_LEN),
              batched=True)
    d = d.rename_column("label", "labels")
    d.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    return d

def insert_word_all(df, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
    return df

def insert_sentence_all(df, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df[df["label"] != target_label].copy(deep=True)
    for idx in df.index:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
    return df

word_asr_df = insert_word_all(clean_valid_df, WORD_TRIGGER, TARGET_LABEL)
sent_asr_df = insert_sentence_all(clean_valid_df, SENT_TRIGGER, TARGET_LABEL)

## Step 1 -- Regenerate the 4 poisoned training sets (deterministic, same seed as E2/E3)

In [4]:
def poison_word_trigger_train(df, poison_rate, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def poison_sentence_trigger_train(df, poison_rate, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    candidates = df.index[df["label"] != target_label].tolist()
    n_poison = int(poison_rate * len(df))
    for idx in rng.sample(candidates, min(n_poison, len(candidates))):
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

word_random_df = poison_word_trigger_train(clean_train_df, POISON_RATE_WORD_RANDOM, WORD_TRIGGER, TARGET_LABEL)
sent_random_df = poison_sentence_trigger_train(clean_train_df, POISON_RATE_SENT_RANDOM, SENT_TRIGGER, TARGET_LABEL)

In [5]:
# CBS selection -- reload E1 as surrogate, recompute margins, pick smallest-margin examples
surrogate = AutoModelForSequenceClassification.from_pretrained("./models/e1_clean").to(DEVICE)
surrogate.eval()

def compute_cbs_scores(model, df, target_label, batch_size=128):
    args = TrainingArguments(output_dir="./tmp_score", per_device_eval_batch_size=batch_size, report_to="none")
    trainer = Trainer(model=model, args=args)
    scored_df = df.copy()
    ds = to_hf_dataset(scored_df)
    logits = trainer.predict(ds).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    scored_df["p_true"] = probs[np.arange(len(scored_df)), scored_df["label"].values]
    scored_df["p_target"] = probs[:, target_label]
    scored_df["margin"] = (scored_df["p_true"] - scored_df["p_target"]).abs()
    return scored_df

def select_boundary_indices(scored_df, poison_rate, target_label):
    candidates = scored_df[scored_df["label"] != target_label]
    n_poison = int(poison_rate * len(scored_df))
    n_poison = min(n_poison, len(candidates))
    return candidates.sort_values("margin", ascending=True).head(n_poison).index

scored_train_df = compute_cbs_scores(surrogate, clean_train_df, TARGET_LABEL)
boundary_idx_word = select_boundary_indices(scored_train_df, POISON_RATE_WORD_CBS, TARGET_LABEL)
boundary_idx_sent = select_boundary_indices(scored_train_df, POISON_RATE_SENT_CBS, TARGET_LABEL)

def apply_word_trigger(df, indices, trigger_word, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_word)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

def apply_sentence_trigger(df, indices, trigger_sentence, target_label, seed=SEED):
    rng = random.Random(seed)
    df = df.copy(deep=True); df["is_poisoned"] = 0
    for idx in indices:
        words = df.at[idx, "sentence"].split()
        pos = rng.randint(0, len(words))
        words.insert(pos, trigger_sentence)
        df.at[idx, "sentence"] = " ".join(words)
        df.at[idx, "label"] = target_label
        df.at[idx, "is_poisoned"] = 1
    return df

word_cbs_df = apply_word_trigger(clean_train_df, boundary_idx_word, WORD_TRIGGER, TARGET_LABEL)
sent_cbs_df = apply_sentence_trigger(clean_train_df, boundary_idx_sent, SENT_TRIGGER, TARGET_LABEL)

CONFIGS = {
    "word_random": {"df": word_random_df, "teacher_dir": "./models/e2_word_trigger", "asr_df": word_asr_df, "poison_rate": POISON_RATE_WORD_RANDOM},
    "word_cbs":    {"df": word_cbs_df,    "teacher_dir": "./models/e3_cbs_word",    "asr_df": word_asr_df, "poison_rate": POISON_RATE_WORD_CBS},
    "sent_random": {"df": sent_random_df, "teacher_dir": "./models/e2_sent_trigger", "asr_df": sent_asr_df, "poison_rate": POISON_RATE_SENT_RANDOM},
    "sent_cbs":    {"df": sent_cbs_df,    "teacher_dir": "./models/e3_cbs_sent",    "asr_df": sent_asr_df, "poison_rate": POISON_RATE_SENT_CBS},
}
for name, c in CONFIGS.items():
    print(name, "poisoned:", c["df"]["is_poisoned"].sum())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

word_random poisoned: 40
word_cbs poisoned: 336
sent_random poisoned: 26
sent_cbs poisoned: 134


## Step 2 -- Build a defense-evaluation subsample per config
All poisoned examples + a random clean sample, capped at `DEFENSE_SAMPLE_SIZE`, so ONION/STRIP (which need many forward passes per example) finish in reasonable time. Raise the cap for your final report.

In [6]:
def build_defense_sample(df, sample_size=DEFENSE_SAMPLE_SIZE, seed=SEED):
    rng = np.random.RandomState(seed)
    poisoned = df[df["is_poisoned"] == 1]
    clean = df[df["is_poisoned"] == 0]
    n_clean = max(0, sample_size - len(poisoned))
    clean_sample = clean.sample(n=min(n_clean, len(clean)), random_state=seed)
    return pd.concat([poisoned, clean_sample]).sample(frac=1, random_state=seed)  # shuffle

for name, c in CONFIGS.items():
    c["defense_sample"] = build_defense_sample(c["df"])
    print(name, "defense sample size:", len(c["defense_sample"]), "of which poisoned:",
          c["defense_sample"]["is_poisoned"].sum())

word_random defense sample size: 3000 of which poisoned: 40
word_cbs defense sample size: 3000 of which poisoned: 336
sent_random defense sample size: 3000 of which poisoned: 26
sent_cbs defense sample size: 3000 of which poisoned: 134


## Step 3 -- Defense implementations
### 3a. ONION (perplexity-based word filtering, Qi et al. 2021)
For each sentence: remove one word at a time, score = (perplexity of original) − (perplexity after removing that word). A big positive drop = that word looks like an unnatural insertion. Flag the sentence if its *biggest* drop exceeds the sample's own mean + 2*std (self-referential threshold, no separate dev set needed).

In [7]:
gpt2_tok = GPT2TokenizerFast.from_pretrained("gpt2")
gpt2 = GPT2LMHeadModel.from_pretrained("gpt2").to(DEVICE).eval()

def sentence_perplexity(sentence):
    enc = gpt2_tok(sentence, return_tensors="pt").to(DEVICE)
    if enc["input_ids"].shape[1] < 2:
        return float("inf")
    with torch.no_grad():
        out = gpt2(**enc, labels=enc["input_ids"])
    return torch.exp(out.loss).item()

def onion_score(sentence):
    words = sentence.split()
    if len(words) < 2:
        return 0.0
    base_ppl = sentence_perplexity(sentence)
    drops = []
    for i in range(len(words)):
        reduced = " ".join(words[:i] + words[i+1:])
        drops.append(base_ppl - sentence_perplexity(reduced))
    return max(drops)

def onion_detect(sample_df):
    scores = sample_df["sentence"].apply(onion_score).values
    thresh = scores.mean() + 2 * scores.std()
    flagged = sample_df.index[scores > thresh]
    return set(flagged), scores

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

### 3b. Spectral Signature (Tran et al. 2018) -- outlier detection in the poisoned teacher's [CLS] embedding space

In [8]:
def get_cls_embeddings(model, df, batch_size=64):
    model.eval()
    embs = []
    sentences = df["sentence"].tolist()
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        enc = tokenizer(batch, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.base_model(**enc)
        embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())  # [CLS] token
    return np.concatenate(embs, axis=0)

def spectral_signature_detect(model, sample_df, target_label, poison_rate):
    target_df = sample_df[sample_df["label"] == target_label]
    X = get_cls_embeddings(model, target_df)
    Xc = X - X.mean(axis=0)
    _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
    scores = (Xc @ Vt[0]) ** 2
    n_remove = min(int(1.5 * poison_rate * len(sample_df)), len(target_df) - 1)
    n_remove = max(n_remove, 0)
    flagged_local = np.argsort(scores)[::-1][:n_remove]
    flagged_index = target_df.index[flagged_local]
    return set(flagged_index), scores

### 3c. STRIP (Gao et al. 2019, text-adapted) -- perturbation entropy

In [9]:
def blend_words(sentence, pool, rng):
    other = rng.choice(pool).split()
    mix = sentence.split() + other[:max(1, len(other)//2)]
    rng.shuffle(mix)
    return " ".join(mix)

def strip_detect(model, sample_df, clean_pool_sentences, n_perturb=6, flag_percentile=25, seed=SEED):
    rng = random.Random(seed)
    model.eval()
    entropies = []
    for sentence in sample_df["sentence"]:
        variants = [blend_words(sentence, clean_pool_sentences, rng) for _ in range(n_perturb)]
        enc = tokenizer(variants, truncation=True, padding="max_length", max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
        mean_p = probs.mean(axis=0)
        entropies.append(-np.sum(mean_p * np.log(mean_p + 1e-12)))
    entropies = np.array(entropies)
    thresh = np.percentile(entropies, flag_percentile)   # LOW entropy = overconfident = suspicious
    flagged = sample_df.index[entropies <= thresh]
    return set(flagged), entropies

### 3d. ABL -- Anti-Backdoor Learning (Li et al. 2021), simplified
Backdoored examples tend to be learned unusually fast (abnormally low loss) early in training. Run one short training epoch on the config's poisoned set, log per-example loss, flag the lowest-loss examples among the currently target-labeled ones.

In [10]:
def abl_detect(train_df, target_label, poison_rate, epochs=3, batch_size=16, lr=2e-5, isolate_percentile=1):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=lr)
    df = train_df.reset_index(drop=False).rename(columns={"index": "orig_index"})
    losses = np.zeros(len(df))
    for epoch in range(epochs):
        for i in range(0, len(df), batch_size):
            batch = df.iloc[i:i+batch_size]
            enc = tokenizer(batch["sentence"].tolist(), truncation=True, padding="max_length",
                             max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
            labels = torch.tensor(batch["label"].values).to(DEVICE)
            logits = model(**enc).logits
            per_ex_loss = F.cross_entropy(logits, labels, reduction="none")
            per_ex_loss.mean().backward()
            opt.step(); opt.zero_grad()
            losses[batch.index.values] = per_ex_loss.detach().cpu().numpy()
    df["loss"] = losses
    target_df = df[df["label"] == target_label]
    thresh = np.percentile(target_df["loss"], isolate_percentile)
    flagged_orig_idx = set(target_df[target_df["loss"] <= thresh]["orig_index"])
    return flagged_orig_idx, df.set_index("orig_index")["loss"]

## Step 4 -- Run all 4 defenses on all 4 configs, get detection metrics

In [11]:
def detection_metrics(flagged_set, df):
    y_true = df["is_poisoned"].values
    y_pred = df.index.isin(flagged_set).astype(int)
    tp = ((y_true == 1) & (y_pred == 1)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    n_pos = (y_true == 1).sum()
    n_neg = (y_true == 0).sum()
    return {"detection_rate": tp / max(n_pos, 1), "false_positive_rate": fp / max(n_neg, 1)}

print("Starting defense evaluation over", len(CONFIGS), "configs")
detection_results = {}
for name, c in CONFIGS.items():
    sample = c["defense_sample"]
    print(f"[{name}] sample size={len(sample)} poisoned={int(sample['is_poisoned'].sum())}")
    print(f"[{name}] loading teacher from {c['teacher_dir']}")
    teacher = AutoModelForSequenceClassification.from_pretrained(c["teacher_dir"]).to(DEVICE)

    print(f"[{name}] running ONION")
    onion_flag, _ = onion_detect(sample)
    print(f"[{name}] ONION flagged {len(onion_flag)} examples")

    print(f"[{name}] running Spectral Signature")
    ss_flag, _ = spectral_signature_detect(teacher, sample, TARGET_LABEL, c["poison_rate"])
    print(f"[{name}] Spectral Signature flagged {len(ss_flag)} examples")

    print(f"[{name}] running STRIP")
    strip_flag, _ = strip_detect(teacher, sample, clean_train_df["sentence"].tolist())
    print(f"[{name}] STRIP flagged {len(strip_flag)} examples")

    print(f"[{name}] running ABL on full poisoned train set")
    abl_flag, _ = abl_detect(c["df"], TARGET_LABEL, c["poison_rate"])  # runs on full poisoned train set
    print(f"[{name}] ABL flagged {len(abl_flag)} examples")

    c["flags"] = {"ONION": onion_flag, "Spectral Signature": ss_flag, "STRIP": strip_flag, "ABL": abl_flag}
    detection_results[name] = {def_name: detection_metrics(flag_set, sample if def_name != "ABL" else c["df"])
                                for def_name, flag_set in c["flags"].items()}
    print(f"[{name}] metrics:", detection_results[name])
    print(name, "done")

detection_table = pd.DataFrame({(name, metric): {d: detection_results[name][d][metric] for d in ["ONION","Spectral Signature","STRIP","ABL"]}
                                 for name in CONFIGS for metric in ["detection_rate","false_positive_rate"]})
detection_table

Starting defense evaluation over 4 configs
[word_random] sample size=3000 poisoned=40
[word_random] loading teacher from ./models/e2_word_trigger


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


[word_random] running ONION


c:\Users\Akshar\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:191: RuntimeWarning: invalid value encountered in subtract
  x = asanyarray(arr - arrmean)


...

[sent_cbs] ONION flagged 0 examples
[sent_cbs] running Spectral Signature
[sent_cbs] Spectral Signature flagged 9 examples
[sent_cbs] running STRIP
[sent_cbs] STRIP flagged 750 examples
[sent_cbs] running ABL on full poisoned train set


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[sent_cbs] ABL flagged 378 examples
[sent_cbs] metrics: {'ONION': {'detection_rate': np.float64(0.0), 'false_positive_rate': np.float64(0.0)}, 'Spectral Signature': {'detection_rate': np.float64(0.0), 'false_positive_rate': np.float64(0.00314026517794836)}, 'STRIP': {'detection_rate': np.float64(0.1044776119402985), 'false_positive_rate': np.float64(0.2568039078855548)}, 'ABL': {'detection_rate': np.float64(0.0), 'false_positive_rate': np.float64(0.0056237446998437845)}}
sent_cbs done


word_random                           word_cbs  \
                   detection_rate false_positive_rate detection_rate   
ONION                       0.000            0.000000       0.000000   
Spectral Signature          0.000            0.000676       0.000000   
STRIP                       0.875            0.241554       0.955357   
ABL                         0.000            0.005601       0.029762   

                                          sent_random                      \
                   false_positive_rate detection_rate false_positive_rate   
ONION                         0.000000            0.0            0.000000   
Spectral Signature            0.008258            0.0            0.000336   
STRIP                         0.161036            0.0            0.252186   
ABL                           0.005521            0.0            0.005615   

                         sent_cbs                      
                   detection_rate false_positive_rate  
ONION                    0.000000            0.000000  
Spectral Signature       0.000000            0.003140  
STRIP                    0.104478            0.256804  
ABL                      0.000000            0.005624

## Step 5 -- The paper-style table: ASR of a model RETRAINED on the filtered training set
'No defense' row = ASR of the already-trained E2/E3 teacher checkpoints (no retraining needed). Every other row: remove the flagged examples from that config's poisoned training set, retrain BERT-base from scratch, measure ASR.

In [12]:
def eval_asr(trainer, asr_df, target_label=TARGET_LABEL):
    d = asr_df.copy(); d["label"] = 0
    logits = trainer.predict(to_hf_dataset(d)).predictions
    preds = np.argmax(logits, axis=-1)
    return float((preds == target_label).mean())

def retrain_and_get_asr(train_df, asr_df, run_name, epochs=RETRAIN_EPOCHS, lr=2e-5, batch_size=16):
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    args = TrainingArguments(output_dir=f"./results_{run_name}", num_train_epochs=epochs,
                              per_device_train_batch_size=batch_size, per_device_eval_batch_size=64,
                              learning_rate=lr, save_strategy="no", logging_steps=500,
                              seed=SEED, report_to="none")
    trainer = Trainer(model=model, args=args, train_dataset=to_hf_dataset(train_df))
    trainer.train()
    return eval_asr(trainer, asr_df)

In [13]:
paper_style_results = {"No defense": {}}

# No-defense row: just eval the already-trained teacher checkpoints, no retraining
for name, c in CONFIGS.items():
    teacher = AutoModelForSequenceClassification.from_pretrained(c["teacher_dir"]).to(DEVICE)
    args = TrainingArguments(output_dir="./tmp_eval", per_device_eval_batch_size=64, report_to="none")
    trainer = Trainer(model=teacher, args=args)
    paper_style_results["No defense"][name] = eval_asr(trainer, c["asr_df"])

for def_name in ["ONION", "Spectral Signature", "STRIP", "ABL"]:
    paper_style_results[def_name] = {}
    for name, c in CONFIGS.items():
        flagged = c["flags"][def_name]
        filtered_df = c["df"][~c["df"].index.isin(flagged)]
        asr = retrain_and_get_asr(filtered_df, c["asr_df"], run_name=f"{def_name}_{name}")
        paper_style_results[def_name][name] = asr
        print(def_name, name, "ASR after filtering+retrain:", asr)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/66971 [00:00<?, ? examples/s]

Step,Training Loss
500,0.341802
1000,0.247231
1500,0.230803
2000,0.218209
2500,0.195678
3000,0.190754
3500,0.196420
4000,0.188922
4500,0.129150
5000,0.124393


Map:   0%|          | 0/428 [00:00<?, ? examples/s]

ABL sent_cbs ASR after filtering+retrain: 0.9579439252336449


In [14]:
final_table = pd.DataFrame(paper_style_results).T[["word_random", "word_cbs", "sent_random", "sent_cbs"]]
final_table = (final_table * 100).round(1)
final_table.columns = ["WordInsert+Random", "WordInsert+CBS", "InsertSent+Random", "InsertSent+CBS"]
os.makedirs("./results", exist_ok=True)
final_table.to_json("./results/e7_defense_table.json")
final_table

,WordInsert+Random,WordInsert+CBS,InsertSent+Random,InsertSent+CBS
No defense,90.9,95.8,98.4,97.2
ONION,89.0,98.6,97.9,99.3
Spectral Signature,70.6,95.1,98.6,91.8
STRIP,9.6,8.6,100.0,91.6
ABL,72.2,99.3,98.4,95.8


## Step 6 -- Students (E5/E6): why the table above doesn't apply, and what to run instead
The students were distilled on **clean, untriggered data** -- there is no poisoned training data for ONION/Spectral Signature/ABL to filter. A data-filtering defense here would correctly find nothing, which isn't an interesting result, it's a category mismatch.

What *is* meaningful for the students: an **inference-time** defense, since a real deployer only has the final student model, not its training data. Below: run STRIP directly on the student's predictions over the triggered eval set, and check whether rejecting the low-entropy (flagged-as-suspicious) predictions lowers the *effective* ASR an end user would experience.

**Prerequisite:** run `e5_distill_random.ipynb` and `e6_distill_cbs.ipynb` first.

In [ ]:
STUDENT_CONFIGS = {
    "word_random_student": {"dir": "./models/e5_random_word_student", "asr_df": word_asr_df},
    "word_cbs_student":    {"dir": "./models/e6_cbs_word_student",    "asr_df": word_asr_df},
    "sent_random_student": {"dir": "./models/e5_random_sent_student", "asr_df": sent_asr_df},
    "sent_cbs_student":    {"dir": "./models/e6_cbs_sent_student",    "asr_df": sent_asr_df},
}

student_strip_results = {}
for name, c in STUDENT_CONFIGS.items():
    student = AutoModelForSequenceClassification.from_pretrained(c["dir"]).to(DEVICE)
    flagged, entropies = strip_detect(student, c["asr_df"], clean_train_df["sentence"].tolist())
    kept = c["asr_df"][~c["asr_df"].index.isin(flagged)]
    args = TrainingArguments(output_dir="./tmp_eval2", per_device_eval_batch_size=64, report_to="none")
    trainer = Trainer(model=student, args=args)
    raw_asr = eval_asr(trainer, c["asr_df"])
    effective_asr = eval_asr(trainer, kept) if len(kept) else float("nan")
    student_strip_results[name] = {"raw_ASR": raw_asr, "flag_rate": len(flagged)/len(c["asr_df"]),
                                    "effective_ASR_after_rejecting_flagged": effective_asr}
    print(name, student_strip_results[name])

pd.DataFrame(student_strip_results).T

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

Map:   0%|          | 0/321 [00:00<?, ? examples/s]

word_random_student {'raw_ASR': 0.09345794392523364, 'flag_rate': 0.25, 'effective_ASR_after_rejecting_flagged': 0.1059190031152648}


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

...

Map:   0%|          | 0/428 [00:00<?, ? examples/s]

Map:   0%|          | 0/321 [00:00<?, ? examples/s]

sent_cbs_student {'raw_ASR': 0.07710280373831775, 'flag_rate': 0.25, 'effective_ASR_after_rejecting_flagged': 0.102803738317757}


,raw_ASR,flag_rate,effective_ASR_after_rejecting_flagged
word_random_student,0.093458,0.25,0.105919
word_cbs_student,0.105140,0.25,0.118380
sent_random_student,0.072430,0.25,0.096573
sent_cbs_student,0.077103,0.25,0.102804


: 